<a href="https://colab.research.google.com/github/AgacheM/Analyzing-Toronto-Airbnb-Dataset/blob/main/CIND_820_Notebook_1_Exploratory_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Import Libraries**

In [ ]:
#Check the version of Python being run
!python --version

# Install all dependencies
!pip install pandas matplotlib seaborn ydata-profiling sweetviz ipython numpy

#Import Libraries
#Data Manipulation, Cleaning, and Analysis
import pandas as pd
import numpy as np

#Data Types
!pip install ydata-profiling
from ydata_profiling import ProfileReport

#Univarate Analysis
!pip install sweetviz
import sweetviz as sv

#Formatting and Visualizations
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from IPython.display import FileLink
from IPython.display import IFrame

#(Colab Specific) Local HTML File Download
from google.colab import files

#Format without scientific notation
pd.options.display.float_format = '{:,.0f}'.format

# **2. Download Listings & Reviews Data from Insideairbnb.com**

In [ ]:
#Import listings files
url_toronto = "https://data.insideairbnb.com/canada/on/toronto/2025-11-11/data/listings.csv.gz"
df_toronto = pd.read_csv(url_toronto, compression='gzip')

url_ottawa = "https://data.insideairbnb.com/canada/on/ottawa/2025-09-22/data/listings.csv.gz"
df_ottawa = pd.read_csv(url_ottawa, compression='gzip')

url_montreal = "https://data.insideairbnb.com/canada/qc/montreal/2025-09-18/data/listings.csv.gz"
df_montreal = pd.read_csv(url_montreal, compression='gzip')

#Add city labels
df_toronto['city'] = 'Toronto'
df_ottawa['city'] = 'Ottawa'
df_montreal['city'] = 'Montreal'

#Combine data
df_listings = pd.concat([df_toronto, df_ottawa, df_montreal], ignore_index=True)

#Encode city labels for future modelling

df_listings['city_code'] = df_listings['city'].map({
    'Toronto': 1,
    'Ottawa': 2,
    'Montreal': 3
})

#3. Check Common Features in Combined Datasets

In [ ]:
#Compare number of columns to ensure they are the same between cities
print("Toronto columns:", len(df_toronto.columns))
print("Ottawa columns:", len(df_ottawa.columns))
print("Montreal columns:", len(df_montreal.columns))

#Keep only the common columns, use a set to remove duplicates and ignore order of columns
common_cols = list(
    set(df_toronto.columns) &
    set(df_ottawa.columns) &
    set(df_montreal.columns)
)

print(f"""
Check all 3 cities' datasets contain the same column names = {set(df_toronto.columns) == set(df_ottawa.columns) == set(df_montreal.columns)}
      """)

print(f"""The datsets of the 3 cities share these column columns: {common_cols}
      """)

#Check row counts
print(f"""The number of rows / observations in each city are:
{df_listings['city'].value_counts()}""")

In [ ]:
#Summarize Listings Data
print(f"""
DATASET OVERVIEW
----------------
Rows: {df_listings.shape[0]:,}
Columns: {df_listings.shape[1]}

FIRST 5 ROWS
---------------""")
display(df_listings.head())

In [ ]:
#Calculate Missing Values
print ("Missing Values:")
missing = df_listings.isnull().sum()
missing = missing[missing>0]

missing_percent=(missing/len(df_listings))*100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_percent
})

#View only columns with missing values
missing_summary = missing_summary[missing_summary["Missing Count"]>0]

#Correct Errors
if len(missing_summary) > 0:
    display(
        missing_summary
        .sort_values (by="Missing %", ascending= False)
        .style.format({
            "Missing Count": "{:,.0f}",
            "Missing %": "{:.2f}%"
        })
        .background_gradient(cmap="Reds")
    )
else:
    print ("No Missing Values Found")

In [ ]:
#Drop features with >50% Missing values
print ("Drop features with >50% missing values")

columns_to_drop = missing_summary[missing_summary["Missing %"]>50].index
df_listings = df_listings.drop(columns=columns_to_drop)

print(f"""
Dropped {len(columns_to_drop)} columns with high missing values.
""")
print(f"""Dropped columns: {(columns_to_drop)}
      """)

print ("Remaning Missing Values:\n")
missing = df_listings.isnull().sum()
missing = missing[missing>0]

missing_percent=(missing/len(df_listings))*100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_percent
})

#View only columns with missing values
missing_summary = missing_summary[missing_summary["Missing Count"]>0]

#Correct Errors
if len(missing_summary) > 0:
    display(
        missing_summary
        .sort_values (by="Missing %", ascending= False)
        .style.format({
            "Missing Count": "{:,.0f}",
            "Missing %": "{:.2f}%"
        })
        .background_gradient(cmap="Reds")
    )
else:
    print ("No Missing Values Found")

# 2. Univarate Analsysis: Data Types and Skewness

In [ ]:
#YData Profiling Report for Automatic Classification of Data Types
print(f"""
NOTE: Although YData Profiling infers data types, manual validation was performed to ensure the correct classification of discrete or continuous variables,
as the automated tool may misinterpret identifies or categorical features that are encoded.

We will correct classification of features in the next section of code.
""")

profile = ProfileReport(
    df_listings,
    title = "Toronto Airbnb Listings - Univariate Exploratory Data Analyis",
    explorative = True
)

output_file = "airbnb_univariate_profile.html"
profile.to_file(output_file)

print(f"Profile report saved as {output_file}")

files.download("airbnb_univariate_profile.html")

In [ ]:
#Univarate Analysis focusing on Skewness of Features
report = sv.analyze(df_listings)
report.show_html("sweetviz_airbnb.html", open_browser= False)
files.download("sweetviz_airbnb.html")

# 3. Manual Feature Selection: Drop Columns Containing PII or not Adding Analytical Value

In [ ]:
more_columns_to_drop = [
    #IDs
    'id', 'host_id', 'scrape_id',
    #URLs or images
    'listing_url', 'host_url', 'picture_url', 'host_thumbnail_url', 'host_picture_url',
    #Text fields
    'name', 'description', 'neighborhood_overview','host_about', 'amenities',
    #Host info or PII
    'host_name', 'host_location', 'host_neighbourhood','host_verifications',
    #Metadata
    'last_scraped', 'calendar_last_scraped','calendar_updated', 'source',
    #Redundant counts
    'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms',
    #Low Analytical Value
    'host_has_profile_pic', 'host_identity_verified']

df_listings = df_listings.drop(columns=more_columns_to_drop, errors='ignore')

print (f"""Dropped {len(more_columns_to_drop)} columns representing PII or not adding analytical value.""")


# 4. Data Cleaning: Nulls

In [ ]:
df_listings['number_of_reviews'] = df_listings['number_of_reviews'].fillna(0)
df_listings['number_of_reviews'] = df_listings['number_of_reviews'].astype(int)

# 5. Addressing Univariate Skewness

In [ ]:
#Transform Price from string to numerical, removing "$ and ," characters
df_listings['price'] = df_listings['price'].replace(r'[\$,]', '', regex=True).astype(float)

#Convert dates from string to datetime
df_listings["host_since"] = pd.to_datetime(df_listings["host_since"])
df_listings["first_review"] = pd.to_datetime(df_listings["first_review"])
df_listings["last_review"] = pd.to_datetime(df_listings["last_review"])

#Transform host_response_rate, host_acceptance_rate from string to numerical
df_listings['host_response_rate'] = df_listings["host_response_rate"].str.replace('%','').astype(float)
df_listings['host_acceptance_rate'] = df_listings["host_acceptance_rate"].str.replace('%','').astype(float)

#Convert the 5 True/False boolean columns to 1 or 0:
 #host_is_superhost, host_has_profile_pic, host_identity_verified, has_availability, instant_bookable



# 6. Handle Price: Missing Values and Skewness

In [ ]:
# Step 1: Handle missing values
median_price = df_listings['price'].median()
df_listings['price'] = df_listings['price'].fillna(median_price)

# Step 2: Check skewness
print(f"""Skewness Value of Price = {df_listings['price'].skew():.2f}

Log transformation is applied to reduce skewness in price.
""")

# Step 3: Log transform
df_listings['log price'] = np.log1p(df_listings['price'])

# Step 4: Check results
print("Check the log transformation:\n")
print(df_listings['log price'].head())
print(f"\nLog-transformed skewness = {df_listings['log price'].skew():.2f}")

# Step 5: Plot
sns.histplot(df_listings['log price'], kde=True)
plt.title("Log Price Distribution")
plt.ylabel("Frequency")
plt.show()

# 7. Manual Univariate Analysis with Focus on Skewness for Remaining Features

In [ ]:
#Manual Univarate Analsys with Focus on Skewness for Remaining Features
print(f"""UNIVARATE ANALYSIS OF DISTRIBUTION AND SKEWNESS:

To interpret skewness values, we say a skewness value that is:
• Close or equal to 0 shows highly symmetrical data. The mean is close or equal to the median. The left and right tails are balanced.
The data points are evenly distributed, and any outliers occur with the same frequency and distance on both high and low ends.

• > 1 is strongly right-skewed. The mean > median. The tail is on the right.
Most data points are heavily concentrated at the left (lower end), while outliers pull the tail to the right (higher end).

• < -1 is strongly left-skewed. The mean < median. The tail is on the left.
Most data points are heavily concentrated at the right (higher end), while outliers pull the tail to the left (lower end).
""")

# PRICE
print(f"Skewness of price: {df_listings['price'].skew():.2f}")

plt.figure(figsize=(8, 4))
sns.histplot(df_listings['price'], kde=True)
plt.title("Distribution of Price")
plt.show()

# LOG PRICE
print(f"Skewness of log price: {df_listings['log price'].skew():.2f}")

plt.figure(figsize=(8, 4))
sns.histplot(df_listings['log price'], kde=True)
plt.title("Distribution of Log Price")
plt.show()


# ACCOMMODATES
print(f"Skewness of accommodates: {df_listings['accommodates'].skew():.2f}")

plt.figure(figsize=(8, 4))
sns.histplot(df_listings['accommodates'], kde=True)
plt.title("Distribution of Accommodates")
plt.show()

# BEDROOMS
print(f"Skewness of bedrooms: {df_listings['bedrooms'].skew():.2f}")

plt.figure(figsize=(8, 4))
sns.histplot(df_listings['bedrooms'], kde=True)
plt.title("Distribution of Bedrooms")
plt.show()

# BEDS
print(f"Skewness of beds: {df_listings['beds'].skew():.2f}")

plt.figure(figsize=(8, 4))
sns.histplot(df_listings['beds'], kde=True)
plt.title("Distribution of Beds")
plt.show()

# NUMBER OF REVIEWS
print(f"Skewness of number_of_reviews: {df_listings['number_of_reviews'].skew():.2f}")

plt.figure(figsize=(8, 4))
sns.histplot(df_listings['number_of_reviews'], kde=True)
plt.title("Distribution of Number of Reviews")
plt.show()

# AVAILABILITY
print(f"Skewness of availability_365: {df_listings['availability_365'].skew():.2f}")

plt.figure(figsize=(8, 4))
sns.histplot(df_listings['availability_365'], kde=True)
plt.title("Distribution of Availability (365 Days)")
plt.show()


#City (non-encoded version)
#The distribution below is reflecting the proportion of listings across cities.
print(f"Skewness of city_code: {df_listings['city_code'].skew():.2f}")


plt.figure(figsize=(8, 4))
sns.countplot(x=df_listings['city'])
plt.title("Distribution of Listings by City")
plt.xlabel("City")
plt.ylabel("Count")
plt.show()


# 7. Bivariate Analysis: Correlation Matrix

In [ ]:
#Bivarate Analysis using Correlation Matrix
plt.figure(figsize=(12, 8))
sns.heatmap(df_listings.corr(numeric_only=True), cmap='coolwarm', annot=False)

plt.title("Correlation Matrix of Numerical Features Across Airbnb Listings")
plt.xlabel("Features")
plt.ylabel("Features")
plt.show()